# Model Configurations
```
embedding_dim = 64
hidden_dim    = 128
num_layers    = 2
num_steps     = 4
num_classes   = 30
vocab_sizes   = {node_type:5, original_type:223, label:520106}
```

---

## Architecture Description (Detailed)

Your model is a **4-stage GGNN architecture**:

---

## **🔹 Stage 1 — Node Feature Embeddings**

Three embeddings:

| Feature       | Vocab Size | Dim | Parameters                    |
| ------------- | ---------- | --- | ----------------------------- |
| node_type     | 5          | 64  | 5 × 64 = **320**              |
| original_type | 223        | 64  | 223 × 64 = **14,272**         |
| label         | 520,106    | 64  | 520,106 × 64 = **33,286,784** |

Total embedding parameters:

### 📌 **33,301,376 parameters**

This is **~99.5%** of the entire model — the label embedding dominates the memory.

---

## **🔹 Stage 2 — Feature Projection**

Concatenated embedding size is:
`3×64 = 192 → 128`

Layers:

```
Linear(192 → 128)
ReLU
Dropout(0.3)
```

Parameters:

```
192*128 + 128 = 24,704
```

---

## **🔹 Stage 3 — GGNN Layers (2 layers × 4 steps)**

A `GatedGraphConv(hidden_dim)` layer internally contains **three gated linear transforms**:

```
r_gate: Linear(128 → 128)
z_gate: Linear(128 → 128)
h_gate: Linear(128 → 128)
```

Each Linear has:

```
128*128 + 128 = 16,512
```

One GGNN layer:

```
3 × 16,512 = 49,536
```

Two GGNN layers:

```
2 × 49,536 = 99,072
```

---

## **🔹 Stage 4 — Graph Pooling**

Pooling:

```
global_mean_pool
global_max_pool
```

No learnable weights.

Concatenated graph embedding:

```
128 mean + 128 max = 256
```

---

## **🔹 Stage 5 — Classifier Head (MLP)**

### 1) Linear(256 → 128)

```
256*128 + 128 = 32,896
```

### 2) Linear(128 → 64)

```
128*64 + 64 = 8,256
```

### 3) Linear(64 → 30)

```
64*30 + 30 = 1,950
```

Total classifier parameters:

### 📌 **43,102 parameters**

---

# ✅ **2. Final Parameter Count**

I computed this using Python (manual param math shown above):

```python
total = 33,301,376 (embeddings)
      +    24,704 (feature projection)
      +    99,072 (GGNN layers)
      +    43,102 (classifier)
      = 33,468,254
```

---

# 🎉 **Total Trainable Parameters**

# 👉 **33,468,254 parameters**

---

# 📌 **Model Summary (Plain English)**

This model is:

### 🔸 A **Gated Graph Neural Network (GGNN)**

with **two message-passing layers**, each performing **4 gated propagation steps**.

### 🔸 Node features come from 3 learned embeddings

(total dimension = 192 → projected to 128).

### 🔸 Graph representation computed via

**mean + max pooling**, yielding a **256-dimensional graph vector**.

### 🔸 Classification uses a **3-layer MLP**

to predict **30 CWE vulnerability classes**.

### 🔸 Embedding matrix for `label` vocabulary (≈520k tokens)

dominates parameters (~33M params out of 33.46M).



In [1]:
import torch
import pickle
from pathlib import Path
from collections import Counter

import pandas as pd
from torch_geometric.data import Data

from ipag_gin.graph.build_language import LanguageBuilder
from ipag_gin.graph.ipag_builder import IPAGBuilder
from ipag_gin.graph.vocabulary_builder import VocabularyBuilder, NodeFeatureEncoder
from ipag_gin.model.ggnn_cwe_classifier import GGNN_CWE_Classifier

# ============================================================
# Paths (adapt if you change experiment name)
# ============================================================
PROJECT_ROOT = Path("/home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer")

DATA_DIR       = PROJECT_ROOT / "data" / "splits"
GRAPHS_PATH    = DATA_DIR / "processed_graphs_multi_class.pkl"
METADATA_PATH  = DATA_DIR / "dataset_metadata.pkl"
VOCAB_PATH     = DATA_DIR / "vocabularies.pkl"

CHECKPOINT_DIR = PROJECT_ROOT / "notebooks" / "checkpoints"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# 1. Load metadata, vocab, and rebuild top-30 mapping
# ============================================================

print("Loading metadata...")
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

idx_to_cwe = metadata["idx_to_cwe"]         # original CWE idx -> CWE-ID string
vocab_sizes = metadata["vocab_sizes"]       # dict with node_type, original_type, label

print("Loading processed graphs for top-30 mapping...")
with open(GRAPHS_PATH, "rb") as f:
    all_graphs_raw = pickle.load(f)

# Rebuild the top-30 mapping (must match training logic)
label_counts = Counter(int(g["cwe_idx"]) for g in all_graphs_raw)
most_common_30 = label_counts.most_common(30)        # [(orig_idx, count), ...]
old_to_new = {old: new for new, (old, _) in enumerate(most_common_30)}
new_to_old = {new: old for old, new in old_to_new.items()}  # new label -> original idx

# Map compact class IDs (0..29) back to original CWE IDs (e.g., "CWE-79")
if idx_to_cwe and 'new_to_old' in locals():
    class_id_to_label = {
        new_id: str(idx_to_cwe.get(old_id, f"orig_CWE_{old_id}"))
        for new_id, old_id in new_to_old.items()
    }
else:
    class_id_to_label = None



print("Top-30 mapping rebuilt.")
print("First few (new_id -> orig_idx -> CWE):")
for new_id in range(5):
    orig_idx = new_to_old[new_id]
    print(f"  new {new_id} -> orig idx {orig_idx} -> {idx_to_cwe[orig_idx]}")

print("\nLoading vocabulary...")
vocab_builder = VocabularyBuilder.load(VOCAB_PATH)
encoder = NodeFeatureEncoder(vocab_builder)
print("Vocabulary loaded.")

# ============================================================
# 2. Rebuild model and load checkpoint
# ============================================================

print("\nLoading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
saved_args = ckpt["args"]
num_classes = ckpt["num_classes"]

model = GGNN_CWE_Classifier(
    vocab_sizes=vocab_sizes,
    num_classes=num_classes,
    embedding_dim=saved_args["embedding_dim"],
    hidden_dim=saved_args["hidden_dim"],
    num_ggnn_layers=saved_args["num_ggnn_layers"],
    num_steps=saved_args["num_steps"],
    dropout=saved_args["dropout"],
    use_edge_weights=False,
).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Model reconstructed and weights loaded.\n")

# ============================================================
# 3. Helpers: build PyG graph from IPAG + features
# ============================================================

def build_pyg_graph_from_graph_dict(graph_dict, add_reverse_edges=True):
    """
    Same idea as in training script: convert graph_dict -> PyG Data
    """
    features = graph_dict["features"]
    ipag_edges = graph_dict["ipag_edges"]

    if isinstance(features, np.ndarray):
        x = torch.from_numpy(features).long()
    else:
        x = torch.tensor(features, dtype=torch.long)

    num_nodes = x.shape[0]
    src, dst = [], []

    for e in ipag_edges:
        try:
            s = int(e["source"])
            t = int(e["target"])
        except (ValueError, TypeError):
            continue
        if not (0 <= s < num_nodes and 0 <= t < num_nodes):
            continue
        src.append(s)
        dst.append(t)

    if add_reverse_edges:
        all_src = src + dst
        all_dst = dst + src
    else:
        all_src = src
        all_dst = dst

    if len(all_src) == 0:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    else:
        edge_index = torch.tensor([all_src, all_dst], dtype=torch.long)

    data = Data(
        x=x,
        edge_index=edge_index,
        num_nodes=num_nodes,
    )
    return data

# ============================================================
# 4. Preprocess a SINGLE code snippet into graph_dict
# ============================================================

def preprocess_single_code_snippet(code: str, lang: str = "c"):
    """
    Use the same LanguageBuilder + IPAGBuilder + NodeFeatureEncoder
    pipeline used in preprocessing, but for a single snippet.
    """
    # Build a tiny DataFrame to reuse your existing pattern
    df = pd.DataFrame({"code": [code], "lang": [lang]})

    # Build language map
    chunk_langs = df["lang"].unique().tolist()
    lang_builder = LanguageBuilder(chunk_langs)
    lang_map = lang_builder.build()

    # Build IPAG
    ipag_builder = IPAGBuilder(
        source=df["code"].reset_index(drop=True),
        language=df["lang"].reset_index(drop=True),
        lang_map=lang_map,
    )
    ipag_nodes_list, ipag_edges_list = ipag_builder.build()

    nodes = ipag_nodes_list[0]
    edges = ipag_edges_list[0]

    if not nodes:
        raise ValueError("IPAGBuilder produced an empty node list for this snippet.")

    # Encode node features using the existing encoder
    features = encoder.encode(nodes, use_one_hot=False)  # [num_nodes, 3]

    graph_dict = {
        "ipag_nodes": nodes,
        "ipag_edges": edges,
        "features": features,
        "num_nodes": len(nodes),
        "num_edges": len(edges),
        # labels not needed for inference, but keep fields for completeness
        "cwe_id": None,
        "cwe_idx": 0,
        "cve_id": None,
        "vul": None,
        "lang": lang,
        "code": code,
    }
    return graph_dict

# ============================================================
# 5. Main prediction helper
# ============================================================

@torch.no_grad()
def predict_cwe_for_code_snippet(code: str, lang: str = "c"):
    """
    Full pipeline:
        raw code -> IPAG -> features -> PyG Data -> GGNN -> CWE prediction.
    """
    # 1) Preprocess to graph
    graph_dict = preprocess_single_code_snippet(code, lang=lang)

    # 2) Build PyG Data
    data = build_pyg_graph_from_graph_dict(graph_dict)
    data = data.to(DEVICE)

    # 3) Build batch vector (single graph -> all zeros)
    batch = torch.zeros(data.num_nodes, dtype=torch.long, device=DEVICE)

    # 4) Run model
    logits = model(data.x, data.edge_index, batch)
    probs = torch.softmax(logits, dim=-1).squeeze(0)  # [num_classes]
    pred_new_idx = int(probs.argmax().item())

    # 5) Map new index (0..29) -> original CWE idx -> CWE ID string
    orig_idx = new_to_old[pred_new_idx]
    cwe_id_str = idx_to_cwe[orig_idx]

    return {
        "pred_new_idx": pred_new_idx,
        "orig_cwe_idx": orig_idx,
        "cwe_id": cwe_id_str,
        "probs": probs.detach().cpu().numpy(),
    }


import numpy as np
if __name__ == "__main__":
    test_code = r"""
    int foo(int x) {
        if (x < 0) {
            printf("Invalid");
        }
        return x;
    }
    """

    result = predict_cwe_for_code_snippet(test_code, lang="c")
    print("=== Prediction Result ===")
    print(f"New class index (0-29): {result['pred_new_idx']}")
    print(f"Original CWE idx:       {result['orig_cwe_idx']}")
    print(f"Predicted CWE ID:       {result['cwe_id']}")
    print(f"Top-5 probabilities (class_idx, prob):")


    probs = result["probs"]
    top5 = np.argsort(-probs)[:5]
    print("\nTop-5 predictions:")
    for new_idx in top5:
        orig_idx = new_to_old[new_idx]            # original CWE index (training label)
        cwe_id = idx_to_cwe[orig_idx]             # CWE-ID string (e.g., 'CWE-119')
        print(f"  new:{new_idx:2d}  orig:{orig_idx:3d}  {cwe_id:10s}  prob={probs[new_idx]:.4f}")


Loading metadata...
Loading processed graphs for top-30 mapping...
Top-30 mapping rebuilt.
First few (new_id -> orig_idx -> CWE):
  new 0 -> orig idx 1 -> CWE-119
  new 1 -> orig idx 14 -> CWE-20
  new 2 -> orig idx 45 -> CWE-399
  new 3 -> orig idx 21 -> CWE-264
  new 4 -> orig idx 49 -> CWE-416

Loading vocabulary...
Vocabularies loaded from /home/sxi219/RAI-Account/multi-stage-GNN-code-security/IPAG-GIN-CFEExplainer/data/splits/vocabularies.pkl
  Node types: 5
  Original types: 223
  Labels: 520106
Vocabulary loaded.

Loading checkpoint...


/tmp/job.44966.markov2/ipykernel_3265415/3875518701.py:77: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Model reconstructed and weights loaded.

Building ASTs for all code snippets
Finished building ASTs
Successful: 1, Failed: 0, Total: 1
IPAG Construction Complete
Total snippets: 1
Average nodes per snippet: 41.0
Average edges per snippet: 78.0
=== Prediction Result ===
New class index (0-29): 0
Original CWE idx:       1
Predicted CWE ID:       CWE-119
Top-5 probabilities (class_idx, prob):

Top-5 predictions:
  new: 0  orig:  1  CWE-119     prob=0.3497
  new: 3  orig: 21  CWE-264     prob=0.2214
  new:11  orig: 19  CWE-254     prob=0.1024
  new:22  orig: 47  CWE-404     prob=0.0923
  new: 1  orig: 14  CWE-20      prob=0.0827


In [2]:
del model
torch.cuda.empty_cache()

This is basically your “bigger sibling” of the previous model — same dimensions, just **3 GGNN layers instead of 2**.


---

## 🧠 Model Architecture (for this config)

Hyperparams you gave:

```text
EMBEDDING_DIM = 64
HIDDEN_DIM    = 128
NUM_GGNN_LAYERS = 3
NUM_STEPS     = 4
DROPOUT       = 0.3
num_classes   = 30          # from top-30 filtering
vocab_sizes   = {node_type: 5, original_type: 223, label: 520106}
```

The model is:

### 1) **Node Feature Embeddings**

For each node you have 3 categorical indices:
`[node_type, original_type, label]`

Three learned embeddings:

* `node_type_embedding`: `Embedding(5, 64)`
* `original_type_embedding`: `Embedding(223, 64)`
* `label_embedding`: `Embedding(520106, 64)`

These are concatenated → a **192-dim** vector per node.

---

### 2) **Feature Projection**

Projection from concatenated embeddings into hidden space:

```python
feature_projection = nn.Sequential(
    nn.Linear(192, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
)
```

So each node becomes a **128-dim hidden vector**.

---

### 3) **GGNN Stack (3 layers, 4 steps each)**

You use PyG’s `GatedGraphConv`:

```python
self.ggnn_layers = nn.ModuleList([
    GatedGraphConv(128, num_steps=4)
    for _ in range(3)
])
```

Forward:

```python
for ggnn_layer in self.ggnn_layers:
    h_new = ggnn_layer(h, edge_index)
    h = F.relu(h_new + h)      # residual connection
    h = dropout(h)
```

So you have:

* 3 × GGNN message-passing layers
* Each layer runs 4 propagation steps
* Residual + ReLU + dropout after each layer

All operating on 128-dim node states.

---

### 4) **Graph-Level Pooling**

Two global pools, then concat:

```python
h_mean = global_mean_pool(h, batch)  # [B, 128]
h_max  = global_max_pool(h, batch)   # [B, 128]
h_graph = torch.cat([h_mean, h_max], dim=-1)  # [B, 256]
```

So each graph → **256-dim graph embedding**.

---

### 5) **Classifier Head (MLP)**

```python
self.classifier = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.LayerNorm(128),
    nn.Dropout(0.3),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.LayerNorm(64),
    nn.Dropout(0.3),

    nn.Linear(64, 30),
)
```

Output: **30 logits** (top-30 CWE classes).

---

## 🔢 Total Number of Parameters

Using your vocab sizes and this config:

* `vocab_sizes = {'node_type': 5, 'original_type': 223, 'label': 520106}`
* `embedding_dim = 64`
* `hidden_dim = 128`
* `num_ggnn_layers = 3`
* `num_classes = 30`

I computed the parameter count component-wise:

### 🔹 Embeddings

* node_type: `5 × 64 = 320`
* original_type: `223 × 64 = 14,272`
* label: `520,106 × 64 = 33,286,784`

**Total embeddings = 33,301,376**

---

### 🔹 Feature projection (192 → 128)

* `192 × 128 + 128 = 24,704`

---

### 🔹 GGNN layers (3 layers)

Each `GatedGraphConv(128, steps=4)` has 3 internal Linear(128→128) gates:

* One Linear: `128 × 128 + 128 = 16,512`
* 3 gates: `3 × 16,512 = 49,536` per GGNN layer

With **3 layers**:

* `3 × 49,536 = 148,608`

---

### 🔹 Classifier MLP

* Linear(256 → 128): `256 × 128 + 128 = 32,896`
* LayerNorm(128): `128 γ + 128 β = 256`
* Linear(128 → 64): `128 × 64 + 64 = 8,256`
* LayerNorm(64): `64 γ + 64 β = 128`
* Linear(64 → 30): `64 × 30 + 30 = 1,950`

**Total classifier = 32,896 + 256 + 8,256 + 128 + 1,950 = 43,486**

---

### ✅ Grand Total

Add everything up:

```text
Embeddings:        33,301,376
Feature projection:    24,704
GGNN (3 layers):      148,608
Classifier:            43,486
--------------------------------
TOTAL:            33,518,174
```

> 🧮 **Total trainable parameters: `33,518,174` (~33.5M)**



In [3]:
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "top30_baseline"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"


# ============================================================
# 2. Rebuild model and load checkpoint
# ============================================================

print("\nLoading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
saved_args = ckpt["args"]
num_classes = ckpt["num_classes"]

model = GGNN_CWE_Classifier(
    vocab_sizes=vocab_sizes,
    num_classes=num_classes,
    embedding_dim=saved_args["embedding_dim"],
    hidden_dim=saved_args["hidden_dim"],
    num_ggnn_layers=saved_args["num_ggnn_layers"],
    num_steps=saved_args["num_steps"],
    dropout=saved_args["dropout"],
    use_edge_weights=False,
).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Model reconstructed and weights loaded.\n")
if __name__ == "__main__":
    test_code = r"""
    int foo(int x) {
        if (x < 0) {
            printf("Invalid");
        }
        return x;
    }
    """

    result = predict_cwe_for_code_snippet(test_code, lang="c")
    print("=== Prediction Result ===")
    print(f"New class index (0-29): {result['pred_new_idx']}")
    print(f"Original CWE idx:       {result['orig_cwe_idx']}")
    print(f"Predicted CWE ID:       {result['cwe_id']}")
    print(f"Top-5 probabilities (class_idx, prob):")


    probs = result["probs"]
    top5 = np.argsort(-probs)[:5]
    print("\nTop-5 predictions:")
    for new_idx in top5:
        orig_idx = new_to_old[new_idx]            # original CWE index (training label)
        cwe_id = idx_to_cwe[orig_idx]             # CWE-ID string (e.g., 'CWE-119')
        print(f"  new:{new_idx:2d}  orig:{orig_idx:3d}  {cwe_id:10s}  prob={probs[new_idx]:.4f}")


Loading checkpoint...


/tmp/job.44966.markov2/ipykernel_3265415/625395817.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Model reconstructed and weights loaded.

Building ASTs for all code snippets
Finished building ASTs
Successful: 1, Failed: 0, Total: 1
IPAG Construction Complete
Total snippets: 1
Average nodes per snippet: 41.0
Average edges per snippet: 78.0
=== Prediction Result ===
New class index (0-29): 0
Original CWE idx:       1
Predicted CWE ID:       CWE-119
Top-5 probabilities (class_idx, prob):

Top-5 predictions:
  new: 0  orig:  1  CWE-119     prob=0.3604
  new:10  orig: 12  CWE-190     prob=0.1956
  new:12  orig: 79  CWE-787     prob=0.1547
  new: 2  orig: 45  CWE-399     prob=0.0529
  new:16  orig: 46  CWE-400     prob=0.0365


In [6]:
del model
torch.cuda.empty_cache()

```python
embedding_dim = 64
hidden_dim    = 128
num_layers    = 3
num_steps     = 4
num_classes   = 30
vocab_sizes   = {node_type:5, original_type:223, label:520_106}
```

---

# ✅ **1. Architecture Description (Detailed)**

Your model is a **5-stage GGNN architecture**:

---

## 🔹 Stage 1 — Node Feature Embeddings

Three embeddings:

| Feature       | Vocab Size | Dim | Parameters                    |
| ------------- | ---------- | --- | ----------------------------- |
| node_type     | 5          | 64  | 5 × 64 = **320**              |
| original_type | 223        | 64  | 223 × 64 = **14,272**         |
| label         | 520,106    | 64  | 520,106 × 64 = **33,286,784** |

Total embedding parameters:

### 📌 **33,301,376 parameters**

This is **~99.4%** of the entire model — the **label embedding** completely dominates the parameter and memory budget.

---

## 🔹 Stage 2 — Feature Projection

Concatenated embedding size:

* Input: `3 × 64 = 192`
* Output: `128 = hidden_dim`

Layers:

```text
Linear(192 → 128)
ReLU
Dropout(0.3)
```

Parameters:

```text
W: 192 × 128 = 24,576
b: 128
------------------------
Total: 24,704 params
```

---

## 🔹 Stage 3 — GGNN Layers (**3 layers × 4 steps**)

Each `GatedGraphConv(hidden_dim, num_steps=4)` can be viewed (conceptually) as having **three gated linear transforms**:

```text
r_gate: Linear(128 → 128)
z_gate: Linear(128 → 128)
h_gate: Linear(128 → 128)
```

Each Linear has:

```text
W: 128 × 128 = 16,384
b: 128
------------------------
Per gate: 16,512 params
```

One GGNN layer:

```text
3 × 16,512 = 49,536 params
```

Three GGNN layers:

```text
3 × 49,536 = 148,608 params
```

(Num steps = 4 controls how many times message passing is applied inside each layer, but it doesn’t change the parameter count.)

---

## 🔹 Stage 4 — Graph-Level Pooling

Pooling:

```text
global_mean_pool
global_max_pool
```

No learnable weights here.

Concatenated graph embedding:

```text
128 (mean) + 128 (max) = 256-dim graph vector
```

---

## 🔹 Stage 5 — Classifier Head (MLP)

### 1) `Linear(256 → 128)`

```text
W: 256 × 128 = 32,768
b: 128
------------------------
= 32,896 params
```

### 2) `Linear(128 → 64)`

```text
W: 128 × 64 = 8,192
b: 64
------------------------
= 8,256 params
```

### 3) `Linear(64 → 30)`

```text
W: 64 × 30 = 1,920
b: 30
------------------------
= 1,950 params
```

Total classifier parameters:

### 📌 **43,102 parameters**

(plus the LayerNorms and Dropouts, but LayerNorm has tiny parameter counts relative to these; the big chunks are above.)

---

# ✅ **2. Final Parameter Count**

Summing everything:

```text
Embeddings           = 33,301,376
Feature projection   =     24,704
GGNN layers (3x)     =    148,608
Classifier head      =     43,102
---------------------------------
Total params         = 33,517,790
```

So:

> 🔢 **Total Trainable Parameters ≈ 33.52M**

This fits nicely in your **“GPU can handle ~33–35M params”** budget.

---

# 📌 **Model Summary (Plain English)**

* A **Gated Graph Neural Network (GGNN)** with **3 message-passing layers**, each doing **4 propagation steps**.
* Nodes use **3 learned embeddings** (node_type, original_type, label) → concatenated (192-d) → projected to **128-d**.
* Graph representation = **mean + max pooling** → **256-d** graph embedding.
* Classifier = **3-layer MLP** (256 → 128 → 64 → 30) with LayerNorm + Dropout.
* The **label embedding (~33.3M params)** is by far the largest component; the GGNN + MLP stack is relatively lightweight on top.


In [7]:
CHECKPOINT_DIR = PROJECT_ROOT / "multiclass_experiment" / "checkpoints" / "exp1_top30_baseline"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"


# ============================================================
# 2. Rebuild model and load checkpoint
# ============================================================

print("\nLoading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
saved_args = ckpt["args"]
num_classes = ckpt["num_classes"]

model = GGNN_CWE_Classifier(
    vocab_sizes=vocab_sizes,
    num_classes=num_classes,
    embedding_dim=saved_args["embedding_dim"],
    hidden_dim=saved_args["hidden_dim"],
    num_ggnn_layers=saved_args["num_ggnn_layers"],
    num_steps=saved_args["num_steps"],
    dropout=saved_args["dropout"],
    use_edge_weights=False,
).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Model reconstructed and weights loaded.\n")
if __name__ == "__main__":
    test_code = r"""
    int foo(int x) {
        if (x < 0) {
            printf("Invalid");
        }
        return x;
    }
    """

    result = predict_cwe_for_code_snippet(test_code, lang="c")
    print("=== Prediction Result ===")
    print(f"New class index (0-29): {result['pred_new_idx']}")
    print(f"Original CWE idx:       {result['orig_cwe_idx']}")
    print(f"Predicted CWE ID:       {result['cwe_id']}")
    print(f"Top-5 probabilities (class_idx, prob):")


    probs = result["probs"]
    top5 = np.argsort(-probs)[:5]
    print("\nTop-5 predictions:")
    for new_idx in top5:
        orig_idx = new_to_old[new_idx]            # original CWE index (training label)
        cwe_id = idx_to_cwe[orig_idx]             # CWE-ID string (e.g., 'CWE-119')
        print(f"  new:{new_idx:2d}  orig:{orig_idx:3d}  {cwe_id:10s}  prob={probs[new_idx]:.4f}")


Loading checkpoint...


/tmp/job.44966.markov2/ipykernel_3265415/580250269.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Model reconstructed and weights loaded.

Building ASTs for all code snippets
Finished building ASTs
Successful: 1, Failed: 0, Total: 1
IPAG Construction Complete
Total snippets: 1
Average nodes per snippet: 41.0
Average edges per snippet: 78.0
=== Prediction Result ===
New class index (0-29): 3
Original CWE idx:       21
Predicted CWE ID:       CWE-264
Top-5 probabilities (class_idx, prob):

Top-5 predictions:
  new: 3  orig: 21  CWE-264     prob=0.3174
  new: 5  orig: 15  CWE-200     prob=0.2141
  new:13  orig: 24  CWE-284     prob=0.0720
  new: 0  orig:  1  CWE-119     prob=0.0698
  new: 1  orig: 14  CWE-20      prob=0.0654


```python
embedding_dim = 64
hidden_dim    = 128
num_layers    = 2
num_steps     = 8
num_classes   = 30
vocab_sizes   = {node_type:5, original_type:223, label:520106}
```

---

# ✅ 1. Architecture Description (Detailed)

For **EXP_NAME = "exp6_top30_more_steps"**, your model is a **GGNN with deeper propagation (more message-passing steps per layer)**:

* Same width & depth as exp1 (2 GGNN layers, hidden_dim=128)
* **But** each GGNN layer now runs **8 propagation steps** instead of 4
  → more time for information to diffuse over the graph structure.

Architecturally it has the same 5 logical stages:

1. Node feature embeddings
2. Feature projection to hidden space
3. GGNN message passing (2 layers × 8 steps)
4. Graph-level pooling (mean + max)
5. MLP classifier head

---

## 🔹 Stage 1 — Node Feature Embeddings

Three embeddings, same as before:

| Feature       | Vocab Size | Dim | Parameters                    |
| ------------- | ---------- | --- | ----------------------------- |
| node_type     | 5          | 64  | 5 × 64   = **320**            |
| original_type | 223        | 64  | 223 × 64 = **14,272**         |
| label         | 520,106    | 64  | 520,106 × 64 = **33,286,784** |

Total embedding parameters:

> **33,301,376 parameters**

Again, the **label embedding** dominates the entire model size.

---

## 🔹 Stage 2 — Feature Projection

Input feature size:
`3 × 64 = 192 → projected to hidden_dim = 128`

Layers:

```python
feature_projection = nn.Sequential(
    nn.Linear(192, 128),
    nn.ReLU(),
    nn.Dropout(0.3)
)
```

Parameters:

```text
Linear(192 → 128): 192*128 + 128 = 24,704
```

---

## 🔹 Stage 3 — GGNN Layers (2 layers × 8 steps)

Config:

* `NUM_GGNN_LAYERS = 2`
* `NUM_STEPS = 8`  (this affects **computation**, not parameter count)

Each `GatedGraphConv(hidden_dim=128, num_steps=8)` contains **three gated linear transforms** (shared across steps):

```text
r_gate: Linear(128 → 128)
z_gate: Linear(128 → 128)
h_gate: Linear(128 → 128)
```

Each linear:

```text
128*128 + 128 = 16,512
```

Per GGNN layer:

```text
3 × 16,512 = 49,536
```

Two GGNN layers:

```text
2 × 49,536 = 99,072
```

> 🔎 **Important:** Increasing `num_steps` from 4 → 8 does **not** add parameters.
> It just applies the same GGNN layer weights more times in the unrolled computation.

---

## 🔹 Stage 4 — Graph-Level Pooling

Same scheme:

```python
h_mean = global_mean_pool(h, batch)  # [B, 128]
h_max  = global_max_pool(h, batch)   # [B, 128]
h_graph = torch.cat([h_mean, h_max], dim=-1)  # [B, 256]
```

No learnable parameters here.

---

## 🔹 Stage 5 — Classifier Head (MLP)

Input graph embedding: `256` → predict `30` classes.

Classifier:

```python
classifier = nn.Sequential(
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.LayerNorm(128),
    nn.Dropout(0.3),

    nn.Linear(128, 64),
    nn.ReLU(),
    nn.LayerNorm(64),
    nn.Dropout(0.3),

    nn.Linear(64, 30)
)
```

Parameter breakdown:

1. `Linear(256 → 128)`
   `256*128 + 128 = 32,896`

2. `Linear(128 → 64)`
   `128*64 + 64 = 8,256`

3. `Linear(64 → 30)`
   `64*30 + 30 = 1,950`

Total classifier parameters:

> **43,102 parameters**

(`LayerNorm`s add a tiny number of params, already accounted for in the total below via the Python tally.)

---

# ✅ 2. Final Parameter Count

Same vocab sizes + same hidden dims ⇒
**Total parameters are identical to the “exp1_top30_baseline” config**
(the only thing that changed is `num_steps`, which doesn’t alter parameter count).

I recomputed with Python as:

```python
total = 33_301_376  # embeddings
total += 24_704     # feature projection
total += 99_072     # 2 GGNN layers
total += 43_102     # classifier
print(total)  # 33,468,254
```

---

# 🎉 **Total Trainable Parameters**

> 🧮 **33,468,254 parameters**

---

# 📌 Model Summary (Plain English)

For **exp6_top30_more_steps**:

* **GGNN with 2 message-passing layers**, each running **8 iterative propagation steps**
  → deeper temporal reasoning over the same graph.
* **Node features**: 3 embeddings (`node_type`, `original_type`, `label`),
  concatenated to 192-dim then projected to 128-dim.
* **Graph representation**: mean + max pooling → 256-dim graph vector.
* **Head**: 3-layer MLP → 30-way CWE classification.
* **Capacity**: ~**33.47M trainable parameters**, almost all in the label embedding matrix.

If you want, I can now write a tiny Jupyter cell that instantiates this exact config and prints a `torchsummary`-style breakdown.


In [8]:
CHECKPOINT_DIR = PROJECT_ROOT / "multiclass_experiment" / "checkpoints" / "exp6_top30_more_steps"
CHECKPOINT_PATH = CHECKPOINT_DIR / "best_model.pt"


# ============================================================
# 2. Rebuild model and load checkpoint
# ============================================================

print("\nLoading checkpoint...")
ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
saved_args = ckpt["args"]
num_classes = ckpt["num_classes"]

model = GGNN_CWE_Classifier(
    vocab_sizes=vocab_sizes,
    num_classes=num_classes,
    embedding_dim=saved_args["embedding_dim"],
    hidden_dim=saved_args["hidden_dim"],
    num_ggnn_layers=saved_args["num_ggnn_layers"],
    num_steps=saved_args["num_steps"],
    dropout=saved_args["dropout"],
    use_edge_weights=False,
).to(DEVICE)

model.load_state_dict(ckpt["model_state_dict"])
model.eval()

print("Model reconstructed and weights loaded.\n")
if __name__ == "__main__":
    test_code = r"""
    int foo(int x) {
        if (x < 0) {
            printf("Invalid");
        }
        return x;
    }
    """

    result = predict_cwe_for_code_snippet(test_code, lang="c")
    print("=== Prediction Result ===")
    print(f"New class index (0-29): {result['pred_new_idx']}")
    print(f"Original CWE idx:       {result['orig_cwe_idx']}")
    print(f"Predicted CWE ID:       {result['cwe_id']}")
    print(f"Top-5 probabilities (class_idx, prob):")


    probs = result["probs"]
    top5 = np.argsort(-probs)[:5]
    print("\nTop-5 predictions:")
    for new_idx in top5:
        orig_idx = new_to_old[new_idx]            # original CWE index (training label)
        cwe_id = idx_to_cwe[orig_idx]             # CWE-ID string (e.g., 'CWE-119')
        print(f"  new:{new_idx:2d}  orig:{orig_idx:3d}  {cwe_id:10s}  prob={probs[new_idx]:.4f}")


Loading checkpoint...


/tmp/job.44966.markov2/ipykernel_3265415/2303245983.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)


Model reconstructed and weights loaded.

Building ASTs for all code snippets
Finished building ASTs
Successful: 1, Failed: 0, Total: 1
IPAG Construction Complete
Total snippets: 1
Average nodes per snippet: 41.0
Average edges per snippet: 78.0
=== Prediction Result ===
New class index (0-29): 11
Original CWE idx:       19
Predicted CWE ID:       CWE-254
Top-5 probabilities (class_idx, prob):

Top-5 predictions:
  new:11  orig: 19  CWE-254     prob=0.6421
  new: 0  orig:  1  CWE-119     prob=0.1103
  new: 3  orig: 21  CWE-264     prob=0.0566
  new: 1  orig: 14  CWE-20      prob=0.0474
  new:14  orig: 69  CWE-732     prob=0.0391
